In [ ]:
from pathlib import Path
 
from pyterrier.datasets import get_dataset, Dataset
from pyterrier.index import IterDictIndexer
from pyterrier.batchretrieve import BatchRetrieve
from pyterrier.pipelines import Experiment
from ir_measures import nDCG, MAP, RR
 
from ir_axioms.axiom import (
    ArgUC, QTArg, QTPArg, aSL,
    PROX1, PROX2, PROX3, PROX4, PROX5,
    TFC1, TFC3, M_TDC, LEN_M_TDC,
    AND, LEN_AND, M_AND, LEN_M_AND,
    DIV, LEN_DIV,
    STMC1, STMC2,
    LNC1, TF_LNC, LB1,
    REG, ANTI_REG, ASPECT_REG,
    ORIG, VoteAxiom,
)
from ir_axioms.tools import MiddlePivotSelection
from ir_axioms.integrations.pyterrier.utils import inject_pyterrier
from ir_axioms.integrations.pyterrier.transformers import (
    KwikSortReranker,
    AggregatedAxiomaticPreferences,
)
from ir_axioms.integrations.pyterrier.estimator import EstimatorKwikSortReranker
from ir_axioms.integrations.pyterrier.experiment import AxiomaticExperiment
 
from rankllama_reranker import rankllama_reranker

In [ ]:
dataset_name = "msmarco-passage"
dataset: Dataset = get_dataset(f"irds:{dataset_name}")
dataset_train: Dataset = get_dataset(f"irds:{dataset_name}/trec-dl-2019/judged")
dataset_test: Dataset = get_dataset(f"irds:{dataset_name}/trec-dl-2020/judged")
 
cache_dir = Path("cache/")
index_dir = cache_dir / "indices" / dataset_name
results_dir = Path("results/")
results_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
if not index_dir.exists():
    indexer = IterDictIndexer(str(index_dir.absolute()))
    indexer.index(dataset.get_corpus_iter(), fields=["text"])

In [ ]:
bm25 = BatchRetrieve(str(index_dir.absolute()), wmodel="BM25")
 

rankllama = bm25 % 20 >> rankllama_reranker
 

inject_pyterrier(
    index_location=index_dir,
    text_field=None,
    dataset=dataset_name,
)

In [ ]:
def _all_axioms():
    return [
        ArgUC().cached(cache_dir / "ArgUC"),
        QTArg().cached(cache_dir / "QTArg"),
        QTPArg().cached(cache_dir / "QTPArg"),
        aSL().cached(cache_dir / "aSL"),
        LNC1().cached(cache_dir / "LNC1"),
        TF_LNC().cached(cache_dir / "TF_LNC"),
        LB1().cached(cache_dir / "LB1"),
        PROX1().cached(cache_dir / "PROX1"),
        PROX2().cached(cache_dir / "PROX2"),
        PROX3().cached(cache_dir / "PROX3"),
        PROX4().cached(cache_dir / "PROX4"),
        PROX5().cached(cache_dir / "PROX5"),
        REG().cached(cache_dir / "REG"),
        ANTI_REG().cached(cache_dir / "ANTI_REG"),
        ASPECT_REG().cached(cache_dir / "ASPECT_REG"),
        AND().cached(cache_dir / "AND"),
        LEN_AND().cached(cache_dir / "LEN_AND"),
        M_AND().cached(cache_dir / "M_AND"),
        LEN_M_AND().cached(cache_dir / "LEN_M_AND"),
        DIV().cached(cache_dir / "DIV"),
        LEN_DIV().cached(cache_dir / "LEN_DIV"),
        TFC1().cached(cache_dir / "TFC1"),
        TFC3().cached(cache_dir / "TFC3"),
        M_TDC().cached(cache_dir / "M_TDC"),
        LEN_M_TDC().cached(cache_dir / "LEN_M_TDC"),
        STMC1().cached(cache_dir / "STMC1"),
        STMC2().cached(cache_dir / "STMC2"),
    ]
 
 
axiom_names = [
    "ArgUC", "QTArg", "QTPArg", "aSL", "LNC1", "TF_LNC", "LB1",
    "PROX1", "PROX2", "PROX3", "PROX4", "PROX5",
    "REG", "ANTI_REG", "ASPECT_REG",
    "AND", "LEN_AND", "M_AND", "LEN_M_AND", "DIV", "LEN_DIV",
    "TFC1", "TFC3", "M_TDC", "LEN_M_TDC", "STMC1", "STMC2",
]
 

In [ ]:

majority_vote_axiom = VoteAxiom(_all_axioms(), minimum_votes=0.5) | ORIG()
 
kwiksort = bm25 % 20 >> KwikSortReranker(
    axiom=majority_vote_axiom,
    pivot_selection=MiddlePivotSelection(),
    text_field=None,
    verbose=True,
)
 

from sklearn.ensemble import RandomForestClassifier
 
random_forest = RandomForestClassifier(max_depth=3)
kwiksort_random_forest = bm25 % 20 >> EstimatorKwikSortReranker(
    axioms=_all_axioms(),
    estimator=random_forest,
    pivot_selection=MiddlePivotSelection(),
    text_field=None,
    verbose=True,
)
kwiksort_random_forest.fit(dataset_train.get_topics(), dataset_train.get_qrels())


from lightgbm import LGBMRanker
from pyterrier.ltr import apply_learned_model
from statistics import mean
 
features = (
    bm25 % 20
    >> AggregatedAxiomaticPreferences(
        axioms=_all_axioms(),
        aggregations=[
            lambda prefs: sum(p >= 0 for p in prefs) / len(prefs),
            lambda prefs: sum(p == 0 for p in prefs) / len(prefs),
            lambda prefs: sum(p <= 0 for p in prefs) / len(prefs),
        ],
        text_field=None,
        verbose=True,
    )
)
 
lambda_mart = LGBMRanker(
    num_iterations=1000, metric="ndcg", eval_at=[10], importance_type="gain"
)
ltr = features >> apply_learned_model(lambda_mart, form="ltr")
ltr.fit(
    dataset_train.get_topics()[:-5],
    dataset_train.get_qrels(),
    dataset_train.get_topics()[-5:],
    dataset_train.get_qrels(),
)
 

In [ ]:
experiment = Experiment(
    [
        bm25,
        rankllama,
        kwiksort ^ bm25,
        kwiksort_random_forest ^ bm25,
        ltr ^ bm25,
    ],
    dataset_test.get_topics(),
    dataset_test.get_qrels(),
    [nDCG @ 10, RR, MAP],
    [
        "BM25",
        "RankLlama",
        "KwikSort (vote)",
        "KwikSort (ORACLE estimé)",
        "Axiomatic LTR",
    ],
    verbose=True,
)
experiment.sort_values(by="nDCG@10", ascending=False, inplace=True)
experiment.to_csv(results_dir / "experiment_results.csv", index=False)
print(experiment)
 

axiomatic_experiment = AxiomaticExperiment(
    retrieval_systems=[bm25, rankllama],
    names=["BM25", "RankLlama"],
    axioms=_all_axioms(),
    axiom_names=axiom_names,
    topics=dataset_test.get_topics(),
    qrels=dataset_test.get_qrels(),
    depth=10,
    filter_by_qrels=True,
    text_field=None,
    verbose=True,
)
 
axiomatic_experiment.preferences.to_csv(results_dir / "preferences.csv", index=False)
axiomatic_experiment.preference_distribution.to_csv(
    results_dir / "preference_distribution.csv", index=False
)
axiomatic_experiment.preference_consistency.to_csv(
    results_dir / "preference_consistency.csv", index=False
)
 
print(axiomatic_experiment.preference_distribution)
print(axiomatic_experiment.preference_consistency.round(2))
 